# HEILO Seed v1.1: mais pré-treino + ajuste de conversa que não "gruda" em frases prontas

**O que a auditoria da v1.0 mostrou**
- Português fluente: a perda por caractere caiu pela metade em relação à v0.3.
- Mas o pré-treino foi curto (96 min, perplexidade 27) e o ajuste de conversa, com só ~470 exemplos, fez o modelo repetir poucas frases prontas para tudo.

**O que muda na v1.1**
1. **Continua o pré-treino da v1.0** a partir do checkpoint no Drive, sem recomeçar, com um corpus **3× maior** de Wikipédia PT.
2. **Ajuste de conversa misturado:** a cada passo o modelo também lê um pedaço de Wikipédia, para não esquecer o português. A taxa de aprendizado é menor e a parada antecipada continua.
3. Avaliação na mesma régua (`heilo_eval_v1` + sonda). Só vira a ativa se passar na regra de promoção.

**Antes:** GPU T4 e o secret `GITHUB_TOKEN` (o mesmo de antes). Depois, `Ambiente de execução → Executar tudo`. Leva cerca de 3h30. Se o Colab cair, rode tudo de novo: ele continua de onde parou.

In [ ]:
# 1) Configuração
REPO = "hugolindo12/HELIO"
BRANCH = "main"
NOME_VERSAO = "v1.1"
MINUTOS_PRETREINO = 150            # tempo EXTRA de pré-treino
MAX_CARACTERES = 1_200_000_000     # corpus maior (a v1.0 usou 400 milhões)
LOTE = 32
PASSOS_SFT = 3000
import torch, time, os, sys, json, shutil, subprocess
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NENHUMA — ative a GPU!")

In [ ]:
# 2) Drive (mesma pasta da v1.0: tokenizer e checkpoint estão lá)
from google.colab import drive
drive.mount("/content/drive")
PASTA = "/content/drive/MyDrive/heilo_seed_v1"
assert os.path.exists(f"{PASTA}/tokenizer.json"), "Não achei o tokenizer da v1.0 no Drive (pasta heilo_seed_v1)."
assert os.path.exists(f"{PASTA}/pretreino.pt"), "Não achei o checkpoint do pré-treino da v1.0 no Drive."

In [ ]:
# 3) Código da HEILO (sempre a versão mais nova do GitHub)
from google.colab import userdata
TOKEN = userdata.get("GITHUB_TOKEN")
URL = f"https://{TOKEN}@github.com/{REPO}.git"
if not os.path.exists("/content/heilo"):
    subprocess.run(["git", "clone", "-b", BRANCH, URL, "/content/heilo"], check=True)
else:
    subprocess.run("cd /content/heilo && git stash -u -q; git pull -q --rebase", shell=True)
sys.path.insert(0, "/content")
!pip -q install datasets tokenizers
!git -C /content/heilo log --oneline -1

## Corpus maior (usa o MESMO tokenizer da v1.0)

In [ ]:
# 4) Corpus grande (salvo no Drive; pula se já existir)
from pathlib import Path
from heilo.training import seed_v1, pretrain
from heilo.models.seed.tokenizer import BPETokenizer
tok = BPETokenizer.from_json(Path(PASTA, "tokenizer.json").read_text(encoding="utf-8"))
corpus_dir = Path(PASTA) / "corpus_grande"
if not (corpus_dir / "meta.json").exists():
    t0 = time.time()
    meta = pretrain.preparar_corpus(seed_v1.textos_wikipedia(MAX_CARACTERES), tok, corpus_dir,
                                    max_tokens=10**12, log=lambda s: None)
    print(meta, f"{(time.time()-t0)/60:.1f} min")
else:
    print(open(corpus_dir / "meta.json").read())

## Pré-treino continuado

In [ ]:
# 5) Continua do checkpoint da v1.0 (guarda uma cópia dele antes, uma vez só)
ck = Path(PASTA) / "pretreino.pt"
copia = Path(PASTA) / "pretreino_v1.0.pt"
if not copia.exists():
    shutil.copy(ck, copia)
arq_passos = Path(PASTA, "passos_v1.1.txt")
if not arq_passos.exists():
    st = torch.load(copia, map_location="cpu", weights_only=False)
    ultimo = st["hist"][-1]
    seg_passo = ultimo["min"] * 60 / ultimo["passo"]
    total = st["passo"] + int(MINUTOS_PRETREINO * 60 / seg_passo)
    arq_passos.write_text(str(total))
    del st
PASSOS = int(arq_passos.read_text())
print("vai até o passo", PASSOS)
cfg = seed_v1.CONFIG_V1
res_pre = pretrain.pretreinar(cfg, tok, corpus_dir, ck, passos=PASSOS, lote=LOTE, lr=3e-4, lr_min=3e-5,
                              tempo_max_min=MINUTOS_PRETREINO + 20, avaliar_cada=500, salvar_cada=1000)
print("pré-treino:", res_pre["passo"], "passos |", res_pre["hist"][-1])

## Ajuste de conversa misturado + avaliação

In [ ]:
# 6) SFT com texto misturado (não esquece o português, não gruda em frases prontas)
from heilo.training.pipeline import TrainingPipeline
pipe = TrainingPipeline()
split = seed_v1.dividir_sft(pipe)
print(split["manifest"])
modelo = res_pre["modelo"]
res_sft = pretrain.ajustar_conversa(modelo, split["treino"], split["val"], passos=PASSOS_SFT, lote=LOTE,
                                    lr=5e-5, paciencia=5, pasta_corpus=corpus_dir, peso_texto=0.5)
arq = pretrain.exportar(modelo, Path(PASTA) / f"heilo_seed_{NOME_VERSAO}.pt",
                        {"versao": NOME_VERSAO, "base": "v1.0 (pré-treino continuado)",
                         "passos_pretreino": res_pre["passo"], "hist_pretreino": res_pre["hist"], "sft": res_sft,
                         "corpus": json.loads((corpus_dir / "meta.json").read_text())})
print("exportado:", arq, round(arq.stat().st_size / 1e6, 1), "MB")

In [ ]:
# 7) Avaliação na mesma régua + decisão automática
from heilo.training.ciclos import CiclosSeed
ciclos = CiclosSeed(pipe)
dec = seed_v1.registrar(ciclos, arq, NOME_VERSAO, split["manifest"],
                        {"base": "v1.0", "pretreino": {"passos": res_pre["passo"], "minutos": res_pre["minutos"],
                                       "ultimo": res_pre["hist"][-1] if res_pre["hist"] else None},
                         "sft": {k: res_sft[k] for k in ("perda_val_inicial", "melhor_perda_val")},
                         "sft_misturado": True})
print("PROMOVIDA" if dec["promovida"] else "NÃO promovida", "—", dec["motivo"])
from heilo.models.seed.gpt import MiniGPTChat
chat = MiniGPTChat(arquivo=ciclos.vdir / NOME_VERSAO / "heilo_seed.pt")
for q in ["oi, tudo bem?", "quem é você?", "qual é o oposto de quente?", "o que é uma função em python?",
          "Responda só com o número: quanto é 2 mais 2?", "me fala sobre o Brasil", "Se hoje é segunda, amanhã é que dia?"]:
    print("Você:", q, "\nHEILO:", chat.responder([{"role": "user", "content": q}], temperatura=0.7), "\n")

In [ ]:
# 8) Salvar no GitHub (corrigido: guarda as sobras antes do pull, para o push não falhar)
!cd /content/heilo && git config user.email "heilo@colab" && git config user.name "HEILO (Colab)"
!cd /content/heilo && git add -f models/seed/versions/{NOME_VERSAO} data/training/versions.json data/training/eval && git add models/seed/weights && git commit -q -m "HEILO Seed {NOME_VERSAO}: pré-treino continuado + SFT misturado (Colab)" || echo "nada novo"
!cd /content/heilo && git stash -u -q; git pull --rebase && git push && echo "ENVIADO PARA O GITHUB"